# Note:
- The data from the [Chicago Data Portal](https://data.cityofchicago.org/browse?category=Public+Safety&sortBy=most_accessed&page=1&pageSize=20) and Crime Data set were enriched using multiple datasets from the portal. We initially stored them in the PostgreSQL database to generate the enriched dataset by joining multiple datasets using the_geom, but we discovered inconsistencies in the police beat, district, and sector fields. All missing fields were determined using multiple fields to generate the most accurate information, but there may be errors during the data wrangling process.

- We designate the primary Crime dataset as the authoritative source of truth. To ensure consistency and address missing values, we perform internal imputation using data from other sources to fill corresponding NaN entries in location-based fields.

In [1]:
# import libraries
from platform import python_version
import sys
import time
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import numpy as np
import re

# python source path
sys.path.append('../Src/')

# random seed
SEED = 1776

# python
import utils
import geo
import geo_dict

# Create a dictionary of versions
versions = {
    "Python": sys.version.split()[0],
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Pyarrow": pa.__version__
}

# Display as a clean DataFrame
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)

# Use a single Arrow string & int64 type instance to save memory
arrow_string = pd.ArrowDtype(pa.string())
arrow_float64 = pd.ArrowDtype(pa.float64())
arrow_int64 = pd.ArrowDtype(pa.int64())
arrow_int8 = pd.ArrowDtype(pa.int8())

# for low cardinality categorical columns (2–128 unique values)
arrow_cat8  = pd.ArrowDtype(pa.dictionary(
                  index_type=pa.int8(), 
                  value_type=pa.string()
              ))

# capture time
start = time.time()

   Library Version
0   Python  3.13.9
1   Pandas   2.3.3
2    NumPy   2.3.4
3  Pyarrow  22.0.0


## Read Data
- The Chicago Crime Data contains Crime, Arrest, IUCR, Neighborhood, and Police Beat datasets from the Chicago Crime Portal.

In [2]:
# display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# reset options
# pd.reset_option('display.max_columns')

In [3]:
# load using pyarrow for performance (crime data is joined between crime & neighborhood & police using geom)
df_crime = pd.read_csv("../Data/chicago_crimes_non_dup_export.csv", engine="pyarrow", dtype_backend="pyarrow")
# Police Info
police = pd.read_csv("../Data/police_beats_export.csv", engine="pyarrow", dtype_backend="pyarrow")

In [4]:
df_crime.head()

,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,sector,ward,community_code,community_name,community_area,fbi_code,x_coordinate,y_coordinate,year,latitude,longitude
0,01G050460,2001-01-24 20:45:00,072XX S RIDGELAND AV,1811,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,t,f,324,3,2,<NA>,<NA>,<NA>,<NA>,18,11890105,18510566,2001,41.764219,-87.582549
1,03J493690,2003-07-12 17:00:00,0105XX S DOBSON AVE,0890,THEFT,FROM BUILDING,APARTMENT,f,f,624,6,2,8,69,GREATER GRAND CROSSING,98853167.7093,06,<NA>,<NA>,2003,<NA>,<NA>
2,04X245238,2004-12-13 21:15:00,006XX N RIDGEWAY AVE,2024,NARCOTICS,POSS: HEROIN(WHITE),SIDEWALK,t,f,1122,11,2,27,23,HUMBOLDT PARK,100480876.502,18,1151273,1903996,2004,41.892451,-87.719888
3,07C115980,2006-03-31 09:15:00,026XX N NARRAGANSETT AVE,0610,BURGLARY,FORCIBLE ENTRY,APARTMENT,f,f,2512,25,1,29,19,BELMONT CRAGIN,109099414.689,05,1133296,1916864,2006,41.928096,-87.78561
4,07HN36467,2007-05-25 14:51:00,022XX N LA CROSSE AVE,1812,NARCOTICS,POSS: CANNABIS MORE THAN 30GMS,RESIDENCE,t,f,2522,25,2,31,19,BELMONT CRAGIN,109099414.689,18,1143697,1914371,2007,41.921066,-87.747452


## Describe Data

In [5]:
df_crime.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,105%,max,std
case_number,8469443,8469443,01G050460,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,8469443,NaN,NaN,NaN,2011-08-03 07:17:49,2001-01-01 00:00:00,2005-06-13 18:51:24,2010-06-06 03:45:00,2017-05-19 13:00:00,2025-12-24 00:00:00,NaN
block,8469443,65509,001XX N STATE ST,17067,NaN,NaN,NaN,NaN,NaN,NaN,NaN
iucr,8469443,418,0820,679332,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_type,8469443,34,THEFT,1798858,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,8469443,569,SIMPLE,996364,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_description,8454105,218,STREET,2213266,NaN,NaN,NaN,NaN,NaN,NaN,NaN
arrest,8469443,2,f,6337467,NaN,NaN,NaN,NaN,NaN,NaN,NaN
domestic,8469443,2,f,7006652,NaN,NaN,NaN,NaN,NaN,NaN,NaN
beat,8469443.0,<NA>,<NA>,<NA>,1183.361997,111.0,621.0,1034.0,1731.0,2535.0,703.10501057


## Data Wrangle
- Chicago's `IUCR` codes (Illinois Uniform Crime Reporting) are four-digit codes for classifying crimes, with the Chicago Police Department (CPD) using over 400, including FBI Index Offenses (homicide, robbery, theft) and Non-Index offenses (vandalism, weapons violations)
- Chicago has `50 wards`, each represented by an alderperson, with boundaries redrawn every eight years
- The Chicago Police Department (CPD) divides the city into `22 Districts`, which are further broken down into smaller patrol zones called `Beats`, with specific 4-digit numbers for each area
- Chicago is divided into `77 official Community Areas.

## The Timeline Definition
* To ensure the analysis is accurate, we define the three eras based on global lockdown patterns:
    * Pre-COVID: January 2001 – February 2020
    * COVID Era: March 2020 – December 31, 2022
    * Post-COVID: January 2023 – Present

In [6]:
# Define boundary timestamps
covid_start      = pd.Timestamp('2020-03-01')
post_covid_start = pd.Timestamp('2023-01-01')

# Define conditions (evaluated top to bottom)
conditions = [
    df_crime['date'] < covid_start,          # Pre-COVID
    df_crime['date'] < post_covid_start      # COVID
]

# Corresponding labels
choices = ['pre_covid', 'covid']

# Apply vectorized selection
df_crime['era'] = (
    np.select(conditions, choices, default='post_covid')
    )

# Cast to memory-efficient categorical dtype
# Only 3 unique values -> int8 index is sufficient
df_crime['era'] = df_crime['era'].astype(arrow_cat8)

# Sanity Check
print(df_crime['era'].value_counts())
print(f"\nEra null count: {df_crime['era'].isna().sum()}")
print(f"\nDate range per era:")
print(df_crime.groupby('era')['date'].agg(['min', 'max']))

era
pre_covid     7092647
post_covid     1052925
covid          623871
Name: count, dtype: int64[pyarrow]

Era null count: 0

Date range per era:
                            min                  max
era                                                 
covid       2020-03-01 00:00:00  2022-12-31 23:55:00
post_covid  2023-01-01 00:00:00  2025-12-24 00:00:00
pre_covid   2001-01-01 00:00:00  2020-02-29 23:59:00


### Invalid x_coordinate and y_coordinate

In [7]:
# Condition 1: Invalid State Plane coordinates
mask_xy = (df_crime.x_coordinate == 0) | (df_crime.y_coordinate == 0)

# Condition 2: Invalid WGS84 coordinates (outside Chicago bounding box)
mask_latlon = (
    (df_crime.latitude  < 41.6)   |
    (df_crime.latitude  > 42.1)   |
    (df_crime.longitude < -87.105) |
    (df_crime.longitude > -87.52)
)

# Diagnostic
print(f"Invalid x/y:      {mask_xy.sum():,}")
print(f"Invalid lat/lon:  {mask_latlon.sum():,}")
print(f"Overlap:          {(mask_xy & mask_latlon).sum():,}")
print(f"Total unique:     {(mask_xy | mask_latlon).sum():,}")

# Null them out
df_crime.loc[mask_xy,     ['x_coordinate', 'y_coordinate']] = pd.NA
df_crime.loc[mask_latlon, ['latitude', 'longitude']]        = pd.NA

Invalid x/y:      149
Invalid lat/lon:  149
Overlap:          149
Total unique:     149


### Invalid community_code

In [8]:
# Invalid community_code
mask = (df_crime.community_code == 0)
mask.sum()

76

In [9]:
# Update community_code
df_crime.loc[mask, ['community_code']] = pd.NA

### Check for Duplicates

In [10]:
# get dupes
dupes = df_crime.duplicated(keep='last')
# any duplicates
if dupes.any():
    print(f"Number of Duplicates: {dupes.sum():,}")
else:
    print("No Duplicates")

No Duplicates


### Unique Values

In [11]:
# display number of unique values
for i in df_crime.columns:
    print(f"{i}: {df_crime[i].nunique():,}")

case_number: 8,469,443
date: 3,545,621
block: 65,509
iucr: 418
primary_type: 34
description: 569
location_description: 218
arrest: 2
domestic: 2
beat: 305
district: 24
sector: 4
ward: 50
community_code: 77
community_name: 77
community_area: 77
fbi_code: 26
x_coordinate: 79,348
y_coordinate: 130,433
year: 25
latitude: 909,235
longitude: 908,633
era: 3


#### location_description

In [12]:
# Replaces : , -, and multi-spaces with a single space
def clean_locations(series):
    out = (
        series.str.replace(r'[\s:,-]+', ' ', regex=True)  # Combined delimiters to space
              .str.replace(r'\s*/\s*', '/', regex=True)   # Fix slashes
              .str.strip()
    )
    
    return out
# Apply regex
df_crime['location_description'] = clean_locations(df_crime['location_description'])

# Dictionary mapping (Vectorized replace)
mapping = {
    'NURSING HOME/RETIREMENT HOME': 'NURSING/RETIREMENT HOME', 
    'OTHER RAILROAD PROP/TRAIN DEPOT': 'OTHER RAILROAD PROPERTY/TRAIN DEPOT',
    'PARKING LOT/GARAGE(NON.RESID.)': 'PARKING LOT/GARAGE (NON RESIDENTIAL)',
    'POLICE FACILITY/VEH PARKING LOT': 'POLICE FACILITY/VEHICLE PARKING LOT',
    'POOLROOM': 'POOL ROOM', 
    'RESIDENCE YARD (FRONT/BACK)': 'RESIDENTIAL YARD (FRONT/BACK)',
    'TAXICAB': 'TAXI CAB',
    'VEHICLE OTHER RIDE SERVICE': 'VEHICLE OTHER RIDE SHARE SERVICE (LYFT UBER ETC.)',
    'VEHICLE OTHER RIDE SHARE SERVICE (E.G. UBER LYFT)': 'VEHICLE OTHER RIDE SHARE SERVICE (LYFT UBER ETC.)'
}

# Apply mapping first
df_crime['location_description'] = df_crime['location_description'].replace(mapping)

### New Feature(s)

In [13]:
# Add Month & Day of the Week
months = ['January', 'February', 'March', 'April', 'May', 'June', 
          'July', 'August', 'September', 'October', 'November', 'December']
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Extract integers and map them
# .dt.month returns 1-12, so we subtract 1 for 0-based indexing
df_crime['month'] = np.array(months)[df_crime['date'].dt.month.values - 1]

# .dt.dayofweek returns 0-6 (0 is Monday)
df_crime['day_of_week'] = np.array(days)[df_crime['date'].dt.dayofweek.values]

# convert to pyarrow
df_crime['month'] = df_crime['month'].astype(arrow_string)
df_crime['day_of_week'] = df_crime['day_of_week'].astype(arrow_string)

In [14]:
# Map to strings first
df_crime['quarter'] = df_crime['date'].dt.quarter.map({1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}).astype(arrow_string)
# combine
df_crime['year_quarter'] = (df_crime['year'].astype("string[pyarrow]") + "-" + df_crime['quarter'])
# Force pyarrow datatype
df_crime['year_quarter'] = df_crime['year_quarter'].astype(arrow_string)

| Interval (Inclusive, Exclusive) | Mathematical Notation | Label        | Hours Included    |
|--------------------------------|----------------------|--------------|-------------------|
| 1st: 0 to 4                    | \([0, 4)\)          | Late Night   | 0, 1, 2, 3       |
| 2nd: 4 to 8                    | \([4, 8)\)          | Early Morning| 4, 5, 6, 7       |
| 3rd: 8 to 12                   | \([8, 12)\)         | Morning      | 8, 9, 10, 11     |
| 4th: 12 to 16                  | \([12, 16)\)        | Afternoon    | 12, 13, 14, 15   |
| 5th: 16 to 20                  | \([16, 20)\)        | Evening      | 16, 17, 18, 19   |
| 6th: 20 to 24                  | \([20, 24)\)        | Night        | 20, 21, 22, 23   |

In [15]:
# Get the hours as a PyArrow-backed integer
hours = df_crime['date'].dt.hour.values

# Use np.digitize for ultra-fast binning (vectorized)
# bins: [0, 4, 8, 12, 16, 20, 24]
# digitize returns 1 for 0-3, 2 for 4-7, etc.
bin_indices = np.digitize(hours, bins=[4, 8, 12, 16, 20])

# Map indices to labels
time_labels = np.array(['Late Night', 'Early Morning', 'Morning', 'Afternoon', 'Evening', 'Night'])
df_crime['time_of_day'] = time_labels[bin_indices]

# Final cast to string[pyarrow]
df_crime['time_of_day'] = df_crime['time_of_day'].astype(arrow_string)

#### FBI Code Mapping

In [16]:
# display fbi_code
print(sorted(df_crime['fbi_code'].unique()))

['01A', '01B', '02', '03', '04A', '04B', '05', '06', '07', '08A', '08B', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '22', '24', '26']


In [17]:
# determine specific values in the data that are missing from the mapping dictionary
df_crime.loc[~df_crime["fbi_code"].isin(geo_dict.fbi_codes.keys()), "fbi_code" ].unique()

<ArrowExtensionArray>
[]
Length: 0, dtype: string[pyarrow]

In [18]:
df_crime[["fbi_code_desc", "fbi_index_code"]] = pd.DataFrame({
    "fbi_code_desc": df_crime["fbi_code"].map(lambda x: geo_dict.fbi_codes[x]["desc"]),
    "fbi_index_code": df_crime["fbi_code"].map(lambda x: geo_dict.fbi_codes[x]["is_index"])
})

# convert to arrow datatype
df_crime["fbi_code_desc"] = df_crime["fbi_code_desc"].astype(arrow_string)
df_crime["fbi_index_code"] = df_crime["fbi_index_code"]

In [19]:
# df_crime[['iucr','primary_description','secondary_description','description','fbi_code']][df_crime['fbi_code'] == '01A'].sample(5)
df_crime[['fbi_code_desc','description', 'location_description', 'domestic', 'fbi_index_code']].sample(n=5, random_state=SEED)

,fbi_code_desc,description,location_description,domestic,fbi_index_code
1667796,Vandalism,TO PROPERTY,OTHER,f,False
3620492,Vandalism,TO VEHICLE,STREET,t,False
3105450,Vandalism,TO VEHICLE,STREET,f,False
4194960,Drug Abuse Violations,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,f,False
6769633,Simple Battery,DOMESTIC BATTERY SIMPLE,APARTMENT,t,False


#### Datatype Change (Boolean)

In [20]:
# check for unique values
df_crime[['arrest','domestic', 'fbi_index_code']].apply(lambda s: s.unique())

,arrest,domestic,fbi_index_code
0,t,f,False
1,f,t,True


In [21]:
# convert to pyarrow boolean
df_crime[['arrest','domestic']] = (
    df_crime[['arrest','domestic']] # domestic: Domestic violence
        .apply(lambda col: col.map({'t': True, 'f': False}))
        .astype(bool)
)

#### Feature Information:
* A Chicago `ward` is one of 50 legislative districts, each represented by an elected Alderman on the City Council, serving as local government branches to provide city services, manage development, and reflect community demographics, with boundaries redrawn every 10 years based on census data.
* The `district` feature refers to the city's 22 police districts, which are geographic areas used to organize crime data.
* The `beat` feature in Chicago crime data identifies the smallest geographic police area (a beat) where a crime occurred.
* The `sector` refers to a specific geographic division used by the Chicago Police Department (CPD), where several smaller `beats` (police patrol areas) are grouped together to form a sector, which then rolls up into a larger `district`, providing a layered geographic context for analyzing crime trends.
* The `Community Area` feature refers to one of 77 distinct, officially defined, and geographically stable neighborhoods used for urban planning and statistical analysis. This feature allows categorizing crime incidents by location, enabling trend analysis and identifying high-crime areas.

In [22]:
df_crime.head()

,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,sector,ward,community_code,community_name,community_area,fbi_code,x_coordinate,y_coordinate,year,latitude,longitude,era,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code
0,01G050460,2001-01-24 20:45:00,072XX S RIDGELAND AV,1811,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,True,False,324,3,2,<NA>,<NA>,<NA>,<NA>,18,11890105,18510566,2001,41.764219,-87.582549,pre_covid,January,Wednesday,Q1,2001-Q1,Night,Drug Abuse Violations,False
1,03J493690,2003-07-12 17:00:00,0105XX S DOBSON AVE,0890,THEFT,FROM BUILDING,APARTMENT,False,False,624,6,2,8,69,GREATER GRAND CROSSING,98853167.7093,06,<NA>,<NA>,2003,<NA>,<NA>,pre_covid,July,Saturday,Q3,2003-Q3,Evening,Larceny – Theft,True
2,04X245238,2004-12-13 21:15:00,006XX N RIDGEWAY AVE,2024,NARCOTICS,POSS: HEROIN(WHITE),SIDEWALK,True,False,1122,11,2,27,23,HUMBOLDT PARK,100480876.502,18,1151273,1903996,2004,41.892451,-87.719888,pre_covid,December,Monday,Q4,2004-Q4,Night,Drug Abuse Violations,False
3,07C115980,2006-03-31 09:15:00,026XX N NARRAGANSETT AVE,0610,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,2512,25,1,29,19,BELMONT CRAGIN,109099414.689,05,1133296,1916864,2006,41.928096,-87.78561,pre_covid,March,Friday,Q1,2006-Q1,Morning,Burglary,True
4,07HN36467,2007-05-25 14:51:00,022XX N LA CROSSE AVE,1812,NARCOTICS,POSS: CANNABIS MORE THAN 30GMS,RESIDENCE,True,False,2522,25,2,31,19,BELMONT CRAGIN,109099414.689,18,1143697,1914371,2007,41.921066,-87.747452,pre_covid,May,Friday,Q2,2007-Q2,Afternoon,Drug Abuse Violations,False


#### Update Datatypes & Fill
- Add padding if required

In [23]:
# display
police.head()

,district,sector,beat
0,1,1,111
1,1,1,112
2,1,1,113
3,1,1,114
4,1,2,121


In [24]:
# change to string and must be three char length
cols = police.columns.to_list()

# iterate cols
for col in cols:

    if col in ['district']:
        # Fill NAs and convert to a standard string for the zfill operation
        # https://www.chicagopolice.org/statistics-data/crime-statistics/
        police[col] = police[col].astype("string").str.zfill(3)
    elif col in ['beat']:
         police[col] = police[col].astype("string").str.zfill(4)
    else:
        # Fill NAs and convert to a standard string
        police[col] = police[col].astype("string")
        
    # Force ArrowDtype
    police[col] = police[col].astype(arrow_string)

police[cols].sample(5)

,district,sector,beat
60,006,1,0612
115,009,3,0935
270,025,3,2532
243,022,1,2213
170,015,1,1511


In [25]:
# change to string and must be three char length
cols = ['district', 'beat', 'ward', 'sector', 'community_code', 'year']

# iterate cols
for col in cols:

    if col in ['district']:
        # Fill NAs and convert to a standard string for the zfill operation
        # https://www.chicagopolice.org/statistics-data/crime-statistics/
        df_crime[col] = df_crime[col].astype("string").str.zfill(3)
    elif col in ['ward', 'community_code']:
        df_crime[col] = df_crime[col].astype("string").str.zfill(2)
    elif col in ['beat']:
         df_crime[col] = df_crime[col].astype("string").str.zfill(4)
    else:
        # Fill NAs and convert to a standard string
        df_crime[col] = df_crime[col].astype("string")
        
    # Force ArrowDtype
    df_crime[col] = df_crime[col].astype(arrow_string)

df_crime[cols].sample(5)

,district,beat,ward,sector,community_code,year
7115969,017,1722,45,2,15,2020
4949920,025,2534,30,3,20,2012
31053093,009,0935,11,3,61,2009
1638694,018,1811,32,1,07,2004
4493137,010,1021,24,2,29,2011


#### Update Invalid district / New column (district_loc)

In [26]:
# display district
utils.wrap_unique(df_crime, 'district')

[001, 002, 003, 004, 005, 006, 007, 008, 009, 010, 011, 012, 014, 015, 016, 017,
018, 019, 020, 021, 022, 024, 025, 031]
::::: Unique Count: 24 (+ 47 nulls)


In [27]:
# Invalid district (21, 31)
mask = df_crime.district.isin(['021', '031'])
mask.sum()

np.int64(281)

In [28]:
# Update Invalid district
df_crime.loc[mask, 'district'] = pd.NA 

In [29]:
# display district
utils.wrap_unique(df_crime, 'district')

[001, 002, 003, 004, 005, 006, 007, 008, 009, 010, 011, 012, 014, 015, 016, 017,
018, 019, 020, 022, 024, 025]
::::: Unique Count: 22 (+ 328 nulls)


In [30]:
utils.wrap_unique(police, 'district')

[001, 002, 003, 004, 005, 006, 007, 008, 009, 010, 011, 012, 014, 015, 016, 017,
018, 019, 020, 022, 024, 025]
::::: Unique Count: 22


In [31]:
utils.wrap_unique(police, 'sector')

[1, 2, 3, 5]
::::: Unique Count: 4


In [32]:
# display sector
utils.wrap_unique(df_crime, 'sector')

[1, 2, 3, 5]
::::: Unique Count: 4 (+ 354,616 nulls)


According to a search, the Chicago Police Department (CPD) currently operates 277 active, specialized beats across 22 districts, employing a community-policing model in which 8–9 officers are assigned to patrol specific areas for at least a year. Our Police table in the Chicago Data Hub lists 274 beats, but our crime data contains 305, and we assume that some beats were consolidated due to Chuicago Police Department overhaul over the past 20 years.

In [33]:
print("Number of Beats in the Crime Data:" , df_crime.beat.nunique())
print("Number of Beats in the Police Data:" , police.beat.nunique())

Number of Beats in the Crime Data: 305
Number of Beats in the Police Data: 274


In [34]:
# Compare
crime_set = set(df_crime.beat.to_list())
police_set = set(police.beat.to_list())

# check sets
print("Does crime_set contain all of police_set: " ,crime_set.issuperset(police_set))
print("Does police_set contain all of crime_set: " ,police_set.issuperset(crime_set))

Does crime_set contain all of police_set:  True
Does police_set contain all of crime_set:  False


In [35]:
# Set Compare (symmetric difference)
print(sorted(crime_set ^ police_set))
print("Mis-match Beat Count:", len(crime_set ^ police_set))

['0134', '0310', '0430', '1311', '1312', '1313', '1322', '1323', '1324', '1331', '1332', '1333', '1650', '2111', '2112', '2113', '2122', '2123', '2124', '2131', '2132', '2133', '2311', '2312', '2313', '2322', '2323', '2324', '2331', '2332', '2333']
Mis-match Beat Count: 31


In [36]:
print(sorted(crime_set - police_set))
print("Extra Beat Count from crime_set:", len(crime_set ^ police_set))

['0134', '0310', '0430', '1311', '1312', '1313', '1322', '1323', '1324', '1331', '1332', '1333', '1650', '2111', '2112', '2113', '2122', '2123', '2124', '2131', '2132', '2133', '2311', '2312', '2313', '2322', '2323', '2324', '2331', '2332', '2333']
Extra Beat Count from crime_set: 31


In [37]:
# Initialize
invalid_beat = list(crime_set - police_set)

In [38]:
# columns to display
cols = ['year', 'community_code', 'community_name', 'community_area', 'ward', 'district', 'sector', 
        'beat', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude']
mask = df_crime.beat.isin(invalid_beat)
print(f"Possible Invalid 'beat' from Cime Data Count:  {(mask.sum()):,}")
bad_beat = df_crime.loc[mask, cols].copy()
bad_beat.head()

Possible Invalid 'beat' from Cime Data Count:  351,331


,year,community_code,community_name,community_area,ward,district,sector,beat,x_coordinate,y_coordinate,latitude,longitude
42,2001,<NA>,<NA>,<NA>,<NA>,002,<NA>,2112,1178109,1882924,41.834059,-87.621973
60,2001,<NA>,<NA>,<NA>,<NA>,002,<NA>,2123,1183840,1878113,41.820725,-87.6010105
70,2001,39,KENWOOD,29071741.9283,04,002,<NA>,2124,1182421,1872504,41.805367,-87.6064105
72,2001,<NA>,<NA>,<NA>,<NA>,002,<NA>,2111,1177107,1890696,41.855409,-87.625414
99,2001,<NA>,<NA>,<NA>,<NA>,019,<NA>,2311,1167921,1930894,41.965917,-87.657969


In [39]:
# count rows by year
bad_beat['year'].value_counts().sort_index()

year
2001    36982
2002    36251
2003    35330
2004    34973
2005    34715
2006    32333
2007    30002
2008    29480
2009    25286
2010    24096
2011    22549
2012     9201
2020        1
2023       91
2024       41
Name: count, dtype: int64[pyarrow]

In [40]:
mask = bad_beat.year.isin(['2024', '2023', '2020'])
bad_beat.loc[mask, cols].drop_duplicates().head()

,year,community_code,community_name,community_area,ward,district,sector,beat,x_coordinate,y_coordinate,latitude,longitude
7789824,2023,76,OHARE,371835607.687,41,016,<NA>,1650,1108491,1934242,41.976182,-87.876421
7807735,2020,76,OHARE,371835607.687,41,016,<NA>,1650,<NA>,<NA>,<NA>,<NA>
7964346,2023,76,OHARE,371835607.687,41,016,<NA>,1650,<NA>,<NA>,<NA>,<NA>
7973848,2024,76,OHARE,371835607.687,41,016,<NA>,1650,1108491,1934242,41.976182,-87.876421
7994678,2024,76,OHARE,371835607.687,41,016,<NA>,1650,1108424,1934249,41.976202,-87.876667


##### According to [Chicago District Map](chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://chicagocop.com/wp-content/uploads/Chicago-Police-Department-Citywide-Area-District-Beat-Map-2009-March.pdf), beat 1650 does not exist; it's possible it was merged into beat 1651, since that is the O'Hare community. We will not change the beat column and will assume it is accurate, since we are unable to verify from the source whether any changes or consolidations occurred with the Chicago Police Department overall.

## **Note:**
* Chicago’s crime data is recorded across a complex framework of overlapping jurisdictions, ranging from political districts to social neighborhoods. At the administrative level, the Chicago Police Department operates through a hierarchy of Districts and Beats. A Beat is the smallest geographic unit, assigned to a specific patrol car for community policing, while multiple Beats are grouped into a District managed by a central precinct. For example, Beats 2511, 2514, and 2521 all fall under the jurisdiction of District 025. Because these boundaries are drawn based on population density and response times rather than cultural history, they rarely align perfectly with the city’s social fabric.
* To provide a more stable lens for analysis, researchers utilize the city’s 77 Community Areas. Established in the 1920s by the University of Chicago, these fixed boundaries remain unchanged by political redistricting or postal updates, allowing for consistent longitudinal tracking of crime trends over decades. In contrast, Chicago’s 50 Wards are political entities redrawn every ten years to ensure equal population representation.
* Ultimately, "neighborhood" designations like "Albany Park" or "Irving Park" reflect social and historical identities rather than law-enforcement jurisdictions. Because these residential areas are often too large for a single patrol car to cover, a single neighborhood is frequently split across multiple Police Beats. This misalignment means that a single criminal incident may be categorized differently depending on whether the analyst is looking through a political (Ward), statistical (Community Area), or operational (Police District) lens.

## **NaN Note:**
- We designate the primary Crime dataset as the authoritative source of truth. To ensure consistency and address missing values, we perform internal imputation by using data from other sources to fill the corresponding NaN entries in the location-based fields.

In [41]:
df_crime.sample(5)

,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,sector,ward,community_code,community_name,community_area,fbi_code,x_coordinate,y_coordinate,year,latitude,longitude,era,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code
2066886,HL352869,2005-05-12 21:48:00,067XX S HAMLIN AVE,0560,ASSAULT,SIMPLE,STREET,False,False,0833,008,3,13,65,WEST LAWN,82315301.6831,08A,1152194,1859825,2005,41.771222,-87.717668,pre_covid,May,Thursday,Q2,2005-Q2,Night,Simple Assault,False
4693021,HT505216,2011-09-20 06:45:00,047XX S LAPORTE AVE,1506,PROSTITUTION,SOLICIT ON PUBLIC WAY,STREET,True,False,0814,008,1,23,56,GARFIELD RIDGE,117890778.429,16,1144155,1872850,2011,41.807118,-87.746811,pre_covid,September,Tuesday,Q3,2011-Q3,Early Morning,Prostitution,False
4241568,HS373593,2010-06-22 10:30:00,014XX W ALBION AVE,2820,OTHER OFFENSE,TELEPHONE THREAT,RESIDENCE,False,True,2432,024,3,40,01,ROGERS PARK,51259902.4506,08A,1165510,1943946,2010,42.001784,-87.66646,pre_covid,June,Tuesday,Q2,2010-Q2,Morning,Simple Assault,False
5863011,HY370135,2015-08-05 13:00:00,004XX N STATE ST,0810,THEFT,OVER $500,PARKING LOT/GARAGE (NON RESIDENTIAL),False,False,1834,018,3,42,08,NEAR NORTH SIDE,7661058105.9728,06,1176302,1903096,2015,41.889453,-87.6279105,pre_covid,August,Wednesday,Q3,2015-Q3,Afternoon,Larceny – Theft,True
7163577,JD309974,2020-07-25 20:50:00,076XX S MARYLAND AVE,1153,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,APARTMENT,False,False,0624,006,2,08,69,GREATER GRAND CROSSING,98853167.7093,11,1183226,1854521,2020,41.1056001,-87.604081,covid,July,Saturday,Q3,2020-Q3,Night,Fraud,False


### NaNs

In [42]:
# display
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,443) ---
                       Count Percentage
ward                  614805    7.2591%
community_code        613748    7.2466%
community_name        613748    7.2466%
community_area        613748    7.2466%
sector                354616    4.1870%
x_coordinate           94452    1.1152%
y_coordinate           94452    1.1152%
latitude               94452    1.1152%
longitude              94452    1.1152%
location_description   15338    0.1811%
district                 328    0.0039%


#### Update x_coordinate & y_coordinate & community information

In [43]:
# columns to display (beat, x_coordinate & y_coordinate provides mapping to community)
cols = ['year', 'community_code', 'community_name', 'community_area', 'ward', 
        'sector', 'district', 'beat', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude']
# mask
mask = (
        df_crime.x_coordinate.isna() & 
        df_crime.y_coordinate.isna() &
        df_crime.community_code.notna() &
        df_crime.ward.notna()
)
# print 
print(f"Missing: {(mask.sum()):,}")
# display
df_crime.loc[mask, cols].sample(n=5, random_state=SEED)

Missing: 85,047


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
7998809,2022,66,CHICAGO LAWN,98279465.1151,16,2,008,0825,<NA>,<NA>,<NA>,<NA>
510511056,2015,43,SOUTH SHORE,81812716.31058,07,2,004,0421,<NA>,<NA>,<NA>,<NA>
7767377,2022,42,WOODLAWN,57815179.512,06,2,003,0321,<NA>,<NA>,<NA>,<NA>
10581254,2019,35,DOUGLAS,46004621.1581,04,1,002,0211,<NA>,<NA>,<NA>,<NA>
7063722,2020,41,HYDE PARK,45105380.1732,05,3,002,0235,<NA>,<NA>,<NA>,<NA>


In [44]:
# Composite key
keys = ['year', 'community_code', 'ward', 'beat']
update_cols = ['x_coordinate', 'y_coordinate', 'latitude', 'longitude']
# New updated DataFrame
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 85,047
Rows updated:       84,127
Rows not updated:   920


In [45]:
# spot check
df_crime.loc[mask, cols].sample(n=5, random_state=SEED)

,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
7998809,2022,66,CHICAGO LAWN,98279465.1151,16,2,008,0825,1161372,1862818,41.77925,-87.683942
510511056,2015,43,SOUTH SHORE,81812716.31058,07,2,004,0421,1196068,1854082,41.1054488,-87.557034
7767377,2022,42,WOODLAWN,57815179.512,06,2,003,0321,1182773,1858454,41.766804,-87.60562
10581254,2019,35,DOUGLAS,46004621.1581,04,1,002,0211,1179474,1882305,41.832329,-87.616983
7063722,2020,41,HYDE PARK,45105380.1732,05,3,002,0235,11810555,1866178,41.787887,-87.587847


In [46]:
# update mask
mask = (
        df_crime.x_coordinate.isna() & 
        df_crime.y_coordinate.isna() &
        df_crime.district.notna() &
        df_crime.community_code.notna() &
        df_crime.ward.notna()
)
# print 
print(f"Missing: {(mask.sum()):,}")
# display
df_crime.loc[mask, cols].sample(n=5, random_state=SEED)

Missing: 920


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
6870733,2013,20,HERMOSA,32602059.4055,26,3,025,2534,<NA>,<NA>,<NA>,<NA>
7724918,2006,60,BRIDGEPORT,58291519.2767,11,1,009,0913,<NA>,<NA>,<NA>,<NA>
7776064,2016,38,GRAND BOULEVARD,48492503.1554,20,2,002,0223,<NA>,<NA>,<NA>,<NA>
6403308,2009,35,DOUGLAS,46004621.1581,04,3,001,0133,<NA>,<NA>,<NA>,<NA>
7125392,2006,58,BRIGHTON PARK,105892790.3114,15,2,009,0921,<NA>,<NA>,<NA>,<NA>


In [47]:
# Composite key
keys = ['year', 'community_code', 'ward', 'district']
update_cols = ['x_coordinate', 'y_coordinate', 'latitude', 'longitude']
# New updated DataFrame
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 920
Rows updated:       680
Rows not updated:   240


In [48]:
# display
df_crime.loc[mask, cols].sample(n=5, random_state=SEED)

,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
6870733,2013,20,HERMOSA,32602059.4055,26,3,025,2534,<NA>,<NA>,<NA>,<NA>
7724918,2006,60,BRIDGEPORT,58291519.2767,11,1,009,0913,1167990,1884680,41.839102,-87.659051
7776064,2016,38,GRAND BOULEVARD,48492503.1554,20,2,002,0223,<NA>,<NA>,<NA>,<NA>
6403308,2009,35,DOUGLAS,46004621.1581,04,3,001,0133,1180326,1885721,41.841683,-87.6131052
7125392,2006,58,BRIGHTON PARK,105892790.3114,15,2,009,0921,<NA>,<NA>,<NA>,<NA>


In [49]:
# update mask
mask = (
        df_crime.x_coordinate.isna() & 
        df_crime.y_coordinate.isna() &
        df_crime.community_code.notna() &
        df_crime.ward.notna()
)
# print 
print(f"Missing: {(mask.sum()):,}")
# display
df_crime.loc[mask, cols].sample(n=5, random_state=SEED)

Missing: 240


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
8376649,2001,32,LOOP,46335565.4586,34,1,001,0112,<NA>,<NA>,<NA>,<NA>
6711428,2009,61,NEW CITY,134636963.254,15,3,009,0931,<NA>,<NA>,<NA>,<NA>
8407716,2001,27,EAST GARFIELD PARK,53883220.8462,28,2,012,1222,<NA>,<NA>,<NA>,<NA>
7992222,2020,19,BELMONT CRAGIN,109099414.689,26,2,025,2522,<NA>,<NA>,<NA>,<NA>
6784639,2007,61,NEW CITY,134636963.254,15,2,009,0924,<NA>,<NA>,<NA>,<NA>


In [50]:
# Composite key
keys = ['year', 'community_code', 'district']
update_cols = ['x_coordinate', 'y_coordinate', 'latitude', 'longitude']
# New updated DataFrame
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 240
Rows updated:       239
Rows not updated:   1


In [51]:
# update mask
mask = (
        df_crime.x_coordinate.isna() & 
        df_crime.y_coordinate.isna() &
        df_crime.community_code.notna() &
        df_crime.ward.notna()
)
# print 
print(f"Missing: {(mask.sum()):,}")
# display
df_crime.loc[mask, cols].head()

Missing: 1


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
7481676,2022,67,WEST ENGLEWOOD,87947691.9478,16,3,009,0932,<NA>,<NA>,<NA>,<NA>


In [52]:
# Composite key
keys = ['year', 'community_code']
update_cols = ['x_coordinate', 'y_coordinate', 'latitude', 'longitude']
# New updated DataFrame
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 1
Rows updated:       1
Rows not updated:   0


#### Ward

In [53]:
# update mask
mask = (df_crime.community_code.notnull() &
        df_crime.district.notnull() &
        df_crime.ward.isna()
       )
# print 
print(f"Missing Ward Info: {(mask.sum()):,}") 
# display
df_crime.loc[mask, cols].sample(n=5, random_state=SEED)

Missing Ward Info: 2,567


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
45427,2001,76,OHARE,371835607.687,<NA>,5,016,1651,1100635,1934208,41.9762,-87.905312
5061054,2001,76,OHARE,371835607.687,<NA>,5,016,1651,1100635,1934208,41.9762,-87.905312
124251,2001,76,OHARE,371835607.687,<NA>,5,016,1651,1100635,1934208,41.9762,-87.905312
541142,2002,76,OHARE,371835607.687,<NA>,5,016,1651,1100658,1934241,41.97629,-87.905227
266896,2001,76,OHARE,371835607.687,<NA>,5,016,1651,1100635,1934208,41.9762,-87.905312


In [54]:
# Composite key
keys = ['year', 'community_code', 'district', 'beat']
update_cols = ['ward']
# Update
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 2,567
Rows updated:       2,564
Rows not updated:   3


In [55]:
df_crime.loc[mask, cols].sample(n=5, random_state=SEED)

,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
45427,2001,76,OHARE,371835607.687,41,5,016,1651,1100635,1934208,41.9762,-87.905312
5061054,2001,76,OHARE,371835607.687,41,5,016,1651,1100635,1934208,41.9762,-87.905312
124251,2001,76,OHARE,371835607.687,41,5,016,1651,1100635,1934208,41.9762,-87.905312
541142,2002,76,OHARE,371835607.687,41,5,016,1651,1100658,1934241,41.97629,-87.905227
266896,2001,76,OHARE,371835607.687,41,5,016,1651,1100635,1934208,41.9762,-87.905312


In [56]:
# update mask
mask = (df_crime.community_code.notnull() &
        df_crime.district.notnull() &
        df_crime.ward.isna()
       )
# print 
print(f"Missing Ward Info: {(mask.sum()):,}") 
# display
df_crime.loc[mask, cols]

Missing Ward Info: 3


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
2948311,2007,05,NORTH CENTER,57054167.85,<NA>,1,016,1611,1121014,1934728,41.977323,-87.830357
2948313,2007,05,NORTH CENTER,57054167.85,<NA>,1,016,1611,1121014,1934728,41.977323,-87.830357
3534129,2008,05,NORTH CENTER,57054167.85,<NA>,1,016,1611,1120937,1934726,41.977319,-87.830641


In [57]:
# Composite key
keys = ['year', 'community_code', 'district']
update_cols = ['ward']
# Update
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 3
Rows updated:       0
Rows not updated:   3


In [58]:
# Composite key
keys = ['year', 'community_code']
update_cols = ['ward']
# Update
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 3
Rows updated:       3
Rows not updated:   0


In [59]:
df_crime.loc[mask, cols].head()

,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
2948311,2007,05,NORTH CENTER,57054167.85,47,1,016,1611,1121014,1934728,41.977323,-87.830357
2948313,2007,05,NORTH CENTER,57054167.85,47,1,016,1611,1121014,1934728,41.977323,-87.830357
3534129,2008,05,NORTH CENTER,57054167.85,32,1,016,1611,1120937,1934726,41.977319,-87.830641


#### Police District & Sector

In [60]:
# update mask
mask = (df_crime.district.isna() &
        df_crime.sector.isna() &
        df_crime.community_code.notna() &
        df_crime.ward.notna()
       )
print(f"NaN district & sector count: {mask.sum()}")
df_crime.loc[mask, cols].drop_duplicates().sample(n=5, random_state=SEED)

NaN district & sector count: 273


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
4020694,2009,68,ENGLEWOOD,85652323.0826,16,<NA>,<NA>,0724,1170402,18621057,41.77889,-87.650839
6738729,2009,20,HERMOSA,32602059.4055,36,<NA>,<NA>,2522,1124178,1931354,41.968013,-87.818796
3611355,2008,19,BELMONT CRAGIN,109099414.689,30,<NA>,<NA>,2514,1136217,1918831,41.933442,-87.774829
1815368,2004,32,LOOP,46335565.4586,42,<NA>,<NA>,0124,1181205,1901996,41.886323,-87.610023
49105892,2012,76,OHARE,371835607.687,41,<NA>,<NA>,1654,1111110,1933291,41.973534,-87.866809


In [61]:
# Composite key
keys = ['year', 'beat', 'community_code', 'ward']
update_cols = ['sector', 'district']
# Update
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 273
Rows updated:       265
Rows not updated:   8


In [62]:
df_crime.loc[mask, cols].drop_duplicates().sample(n=5, random_state=SEED)

,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
4020694,2009,68,ENGLEWOOD,85652323.0826,16,2,007,0724,1170402,18621057,41.77889,-87.650839
6738729,2009,20,HERMOSA,32602059.4055,36,2,025,2522,1124178,1931354,41.968013,-87.818796
3611355,2008,19,BELMONT CRAGIN,109099414.689,30,1,025,2514,1136217,1918831,41.933442,-87.774829
1815368,2004,32,LOOP,46335565.4586,42,2,001,0124,1181205,1901996,41.886323,-87.610023
49105892,2012,76,OHARE,371835607.687,41,5,016,1654,1111110,1933291,41.973534,-87.866809


In [63]:
# update mask
mask = (df_crime.district.isna() &
        df_crime.sector.isna() &
        df_crime.community_code.notna() &
        df_crime.ward.notna()
       )
print(f"NaN district & sector count: {mask.sum()}")
df_crime.loc[mask, cols].head(8)

NaN district & sector count: 8


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
1221127,2003,35,DOUGLAS,46004621.1581,03,<NA>,<NA>,2112,1178113,1884324,41.837901,-87.621915
1498007,2004,35,DOUGLAS,46004621.1581,03,<NA>,<NA>,2112,1178113,1884324,41.837901,-87.621915
1676679,2004,35,DOUGLAS,46004621.1581,03,<NA>,<NA>,2112,1178113,1884324,41.837901,-87.621915
1713510,2004,35,DOUGLAS,46004621.1581,03,<NA>,<NA>,2112,1178113,1884324,41.837901,-87.621915
35105631,2008,28,NEAR WEST SIDE,158492466.554,27,<NA>,<NA>,1333,1163681,1902280,41.887489,-87.674367
3607044,2008,24,WEST TOWN,1210562904.597,27,<NA>,<NA>,1323,1167418,1908128,41.903457,-87.6604105
4346463,2010,76,OHARE,371835607.687,41,<NA>,<NA>,1611,1100260,1946117,42.008885,-87.906473
6738817,2002,20,HERMOSA,32602059.4055,36,<NA>,<NA>,2522,1124178,1931354,41.968013,-87.818796


In [64]:
# Composite key
keys = ['year', 'community_code']
update_cols = ['sector', 'district']
# Update
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 8
Rows updated:       8
Rows not updated:   0


In [65]:
df_crime.loc[mask, cols].head(8)

,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
1221127,2003,35,DOUGLAS,46004621.1581,03,1,002,2112,1178113,1884324,41.837901,-87.621915
1498007,2004,35,DOUGLAS,46004621.1581,03,1,002,2112,1178113,1884324,41.837901,-87.621915
1676679,2004,35,DOUGLAS,46004621.1581,03,1,002,2112,1178113,1884324,41.837901,-87.621915
1713510,2004,35,DOUGLAS,46004621.1581,03,1,002,2112,1178113,1884324,41.837901,-87.621915
35105631,2008,28,NEAR WEST SIDE,158492466.554,27,1,012,1333,1163681,1902280,41.887489,-87.674367
3607044,2008,24,WEST TOWN,1210562904.597,27,3,014,1323,1167418,1908128,41.903457,-87.6604105
4346463,2010,76,OHARE,371835607.687,41,5,016,1611,1100260,1946117,42.008885,-87.906473
6738817,2002,20,HERMOSA,32602059.4055,36,2,025,2522,1124178,1931354,41.968013,-87.818796


In [66]:
# Police Info
df_crime[['sector', 'district']].isna().sum()

sector      354343
district        55
dtype: int64

In [67]:
# update mask
mask = (df_crime.district.isna() &
       df_crime.x_coordinate.notna()
       )
print(f"NaN count: {mask.sum()}")
df_crime.loc[mask, cols].sample(n=10, random_state=SEED)

NaN count: 55


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
4653817,2011,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,11010552,1930183,41.965057,-87.8791053
4643390,2011,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,11010552,1930183,41.965057,-87.8791053
4335362,2010,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,11010552,1930183,41.965057,-87.8791053
5105792,2002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0533,1179972,1814476,41.646187,-87.617227
4353039,2010,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,11010552,1930183,41.965057,-87.8791053
555428,2002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0533,1179972,1814476,41.646187,-87.617227
7820979,2023,01,ROGERS PARK,51259902.4506,<NA>,<NA>,<NA>,2422,1163643,11050346,42.019386,-87.673147
4086637,2010,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,11010552,1930183,41.965057,-87.8791053
4160447,2010,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,1108128,1930710,41.9664105,-87.877825
4085515,2010,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,11010552,1930183,41.965057,-87.8791053


In [68]:
# Composite key
keys = ['year', 'x_coordinate', 'y_coordinate']
update_cols = ['community_code', 'community_name', 'community_area', 'sector', 'ward', 'district']
# Update
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 55
Rows updated:       2
Rows not updated:   53


In [69]:
# update mask
mask = (df_crime.district.isna() &
       df_crime.x_coordinate.notna()
       )
print(f"NaN count: {mask.sum()}")
df_crime.loc[mask, cols].sample(n=10, random_state=SEED)

NaN count: 53


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
5650916,2014,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,1106263,1941939,41.997336,-87.884466
4085515,2010,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,11010552,1930183,41.965057,-87.8791053
5105792,2002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0533,1179972,1814476,41.646187,-87.617227
43852105,2010,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,11010552,1930183,41.965057,-87.8791053
555428,2002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0533,1179972,1814476,41.646187,-87.617227
40910566,2010,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,11010552,1930183,41.965057,-87.8791053
4454199,2010,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,11010552,1930183,41.965057,-87.8791053
42105031,2010,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,11010552,1930183,41.965057,-87.8791053
4599662,2011,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,11010552,1930183,41.965057,-87.8791053
4353039,2010,76,OHARE,371835607.687,<NA>,<NA>,<NA>,1654,11010552,1930183,41.965057,-87.8791053


In [70]:
# Composite key
keys = ['year', 'community_code']
update_cols = ['community_code', 'community_name', 'community_area', 'sector', 'ward', 'district']
# Update
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 53
Rows updated:       41
Rows not updated:   12


In [71]:
df_crime.loc[mask, cols].sample(n=10, random_state=SEED)

,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
5650916,2014,76,OHARE,371835607.687,41,5,016,1654,1106263,1941939,41.997336,-87.884466
4085515,2010,76,OHARE,371835607.687,41,5,016,1654,11010552,1930183,41.965057,-87.8791053
5105792,2002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0533,1179972,1814476,41.646187,-87.617227
43852105,2010,76,OHARE,371835607.687,41,5,016,1654,11010552,1930183,41.965057,-87.8791053
555428,2002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0533,1179972,1814476,41.646187,-87.617227
40910566,2010,76,OHARE,371835607.687,41,5,016,1654,11010552,1930183,41.965057,-87.8791053
4454199,2010,76,OHARE,371835607.687,41,5,016,1654,11010552,1930183,41.965057,-87.8791053
42105031,2010,76,OHARE,371835607.687,41,5,016,1654,11010552,1930183,41.965057,-87.8791053
4599662,2011,76,OHARE,371835607.687,41,1,016,1654,11010552,1930183,41.965057,-87.8791053
4353039,2010,76,OHARE,371835607.687,41,5,016,1654,11010552,1930183,41.965057,-87.8791053


In [72]:
# update mask
mask = (df_crime.district.isna() &
       df_crime.x_coordinate.notna()
       )
print(f"NaN count: {mask.sum()}")
df_crime.loc[mask, cols].head(12)

NaN count: 12


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
544772,2002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1621,1139607,1945673,42.007037,-87.761712
555428,2002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0533,1179972,1814476,41.646187,-87.617227
5105792,2002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0533,1179972,1814476,41.646187,-87.617227
597492,2002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1611,1129345,1943130,42.000241,-87.7910527
606934,2002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1614,1120919,1934726,41.977319,-87.830707
1077346,2003,<NA>,<NA>,<NA>,41,<NA>,<NA>,1651,1111110,1933291,41.973534,-87.866809
2621435,2006,<NA>,<NA>,<NA>,19,<NA>,<NA>,2212,1159640,1832678,41.696576,-87.691116
3028818,2007,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1614,1112570,1936925,41.983484,-87.861366
31105626,2007,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1614,1112570,1936925,41.983484,-87.861366
33810586,2008,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1614,1112570,1936925,41.983484,-87.861366


In [73]:
# Get Index Number
community_district_sector = df_crime.loc[mask, cols].index
len(community_district_sector)

12

### DataFrame Maint

In [74]:
# collapse fragmented blocks
df_crime = df_crime.copy()
# display
df_crime.head()

,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,sector,ward,community_code,community_name,community_area,fbi_code,x_coordinate,y_coordinate,year,latitude,longitude,era,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code
0,01G050460,2001-01-24 20:45:00,072XX S RIDGELAND AV,1811,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,True,False,0324,003,2,<NA>,<NA>,<NA>,<NA>,18,11890105,18510566,2001,41.764219,-87.582549,pre_covid,January,Wednesday,Q1,2001-Q1,Night,Drug Abuse Violations,False
1,03J493690,2003-07-12 17:00:00,0105XX S DOBSON AVE,0890,THEFT,FROM BUILDING,APARTMENT,False,False,0624,006,2,08,69,GREATER GRAND CROSSING,98853167.7093,06,1184198,1855214,2003,41.105788,-87.600498,pre_covid,July,Saturday,Q3,2003-Q3,Evening,Larceny – Theft,True
2,04X245238,2004-12-13 21:15:00,006XX N RIDGEWAY AVE,2024,NARCOTICS,POSS: HEROIN(WHITE),SIDEWALK,True,False,1122,011,2,27,23,HUMBOLDT PARK,100480876.502,18,1151273,1903996,2004,41.892451,-87.719888,pre_covid,December,Monday,Q4,2004-Q4,Night,Drug Abuse Violations,False
3,07C115980,2006-03-31 09:15:00,026XX N NARRAGANSETT AVE,0610,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,2512,025,1,29,19,BELMONT CRAGIN,109099414.689,05,1133296,1916864,2006,41.928096,-87.78561,pre_covid,March,Friday,Q1,2006-Q1,Morning,Burglary,True
4,07HN36467,2007-05-25 14:51:00,022XX N LA CROSSE AVE,1812,NARCOTICS,POSS: CANNABIS MORE THAN 30GMS,RESIDENCE,True,False,2522,025,2,31,19,BELMONT CRAGIN,109099414.689,18,1143697,1914371,2007,41.921066,-87.747452,pre_covid,May,Friday,Q2,2007-Q2,Afternoon,Drug Abuse Violations,False


#### Examine Each NaNs

In [105]:
# NaNs
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,443) ---
                       Count Percentage
community_code        613748    7.2466%
community_name        613748    7.2466%
community_area        613748    7.2466%
ward                  6121105    7.2283%
sector                354300    4.1833%
location_description   15338    0.1811%
x_coordinate            9405    0.1110%
y_coordinate            9405    0.1110%
latitude                9405    0.1110%
longitude               9405    0.1110%
district                  12    0.0001%


#### Check x_coordinate	& y_coordinate & latitude & longitude

In [76]:
# update mask
mask = (df_crime.x_coordinate.isna() &
        df_crime.longitude.notna() &
        df_crime.district.notnull()
       )
print(f"NaN count: {mask.sum()}")
df_crime.loc[mask, cols].head()

NaN count: 0


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude


In [77]:
# Composite key
keys = ['year', 'latitude', 'longitude']
update_cols = ['community_code', 'community_name', 'community_area', 'sector', 'ward', 'x_coordinate', 'y_coordinate']
# Update
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 0
Rows updated:       0
Rows not updated:   0


In [78]:
df_crime.loc[mask, cols].head()

,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude


In [79]:
# Composite key
keys = ['year', 'beat', 'district']
update_cols = ['community_code', 'community_name', 'community_area', 'sector', 'ward', 'x_coordinate', 'y_coordinate']
# Update
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 0
Rows updated:       0
Rows not updated:   0


In [80]:
df_crime.loc[mask, cols].head()

,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude


In [81]:
# update mask
mask = (df_crime.x_coordinate.isna() &
        df_crime.longitude.notna() &
        df_crime.district.notnull()
       )
print(f"NaN count: {mask.sum()}")
df_crime.loc[mask, cols].head()

NaN count: 0


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude


In [82]:
# Composite key
keys = ['year', 'district']
update_cols = ['community_code', 'community_name', 'community_area', 'sector', 'ward', 'x_coordinate', 'y_coordinate']
# Update
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 0
Rows updated:       0
Rows not updated:   0


In [83]:
df_crime.loc[mask, cols].head()

,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude


#### Ward & Community

In [84]:
# update mask
mask = (df_crime.community_code.isna() & 
        df_crime.ward.isna() &
        df_crime.x_coordinate.notna() &
        df_crime.district.notnull() &
        df_crime.sector.notna()
       )
print(f"NaN count: {(mask.sum()):,}")
df_crime.loc[mask, cols].drop_duplicates().sample(n=5, random_state=SEED)

NaN count: 556,920


,year,community_code,community_name,community_area,ward,sector,district,beat,x_coordinate,y_coordinate,latitude,longitude
72017,2001,<NA>,<NA>,<NA>,<NA>,2,005,0524,1170472,1827169,41.68123,-87.651616
48568,2001,<NA>,<NA>,<NA>,<NA>,3,024,2433,1163286,1941702,41.9105674,-87.674705
5910581,2002,<NA>,<NA>,<NA>,<NA>,3,002,0233,1178276,1866596,41.78925,-87.621856
500734,2002,<NA>,<NA>,<NA>,<NA>,3,012,1231,1166902,1894801,41.866898,-87.6621053
5161053,2002,<NA>,<NA>,<NA>,<NA>,2,005,0524,1168071,1828252,41.684254,-87.660374


In [85]:
# Composite key
keys = ['year', 'sector', 'district', 'beat', 'x_coordinate', 'y_coordinate']
update_cols = ['community_code', 'community_name', 'community_area', 'ward']
# Update
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

----- Imputation complete. -----
Total rows in mask: 556,920
Rows updated:       93,385
Rows not updated:   463,535


In [86]:
# update mask
mask = (df_crime.community_code.isna() & 
        df_crime.ward.isna() &
        df_crime.x_coordinate.notna() &
        df_crime.district.notnull() &
        df_crime.sector.notna()
       )
print(f"NaN count: {(mask.sum()):,}")
# Composite key
keys = ['year', 'x_coordinate', 'y_coordinate']
update_cols = ['community_code', 'community_name', 'community_area', 'ward']
# Update
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

NaN count: 463,535
----- Imputation complete. -----
Total rows in mask: 463,535
Rows updated:       3,505
Rows not updated:   460,030


In [87]:
# update mask
mask = (df_crime.community_code.isna() & 
        df_crime.ward.isna() &
        df_crime.x_coordinate.notna() &
        df_crime.district.notnull() &
        df_crime.sector.notna()
       )
print(f"NaN count: {(mask.sum()):,}")
# Composite key
keys = ['x_coordinate', 'y_coordinate']
update_cols = ['community_code', 'community_name', 'community_area', 'ward']
# Update
df_crime = geo.impute_data(df_crime, keys, update_cols, mask)

NaN count: 460,030
----- Imputation complete. -----
Total rows in mask: 460,030
Rows updated:       33,715
Rows not updated:   426,315


#### Update sector

In [88]:
# Invert the mapping
cpd_sector = {
    dist: sector
    for sector, dists in geo_dict.cpd_sector.items() # Outer loop: looping over sector: list_of_districts
    for dist in dists # Inner loop: looping over each district inside that list
}

# Intentional overwrite: sector is derived deterministically from district
# using the cpd_sector dictionary (source:https://www.chicagopolice.org/statistics-data/crime-statistics/)
# sector is represented as Area
# update sector (note: it's not correct for all the years)
df_crime['sector'] = (
    df_crime['district']
        .map(cpd_sector)
).astype(arrow_cat8)

In [89]:
# New column
df_crime['district_location'] = df_crime.district.map(geo_dict.cpd_districts)
df_crime['district_location'] = df_crime['district_location'].astype(arrow_cat8)

In [90]:
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,443) ---
                       Count Percentage
community_code        483143    5.7045%
community_name        483143    5.7045%
community_area        483143    5.7045%
ward                  481590    5.6862%
location_description   15338    0.1811%
x_coordinate            9405    0.1110%
y_coordinate            9405    0.1110%
latitude                9405    0.1110%
longitude               9405    0.1110%
district                  12    0.0001%
sector                    12    0.0001%
district_location         12    0.0001%


In [91]:
# Final State
print(f"Shape: {df_crime.shape}")
print(f"\nDtypes:\n{df_crime.dtypes}")
utils.any_nans(df_crime)

Shape: (8469443, 31)

Dtypes:
case_number                                               string[pyarrow]
date                                                timestamp[s][pyarrow]
block                                                     string[pyarrow]
iucr                                                      string[pyarrow]
primary_type                                              string[pyarrow]
description                                               string[pyarrow]
location_description                                      string[pyarrow]
arrest                                                               bool
domestic                                                             bool
beat                                                      string[pyarrow]
district                                                  string[pyarrow]
sector                  dictionary<values=string, indices=int8, ordere...
ward                                                      string[pyarrow]
communit

* Unresolvable nulls ($\approx$ 5.7% community_code, $\approx$ 5.7% ward,  $\approx$ 0.11% x_coordinate)
* Decision: retain rows - nulls preserved for auditability and future validation.
* Downstream analysis does not depend on community_code or ward features.

## Total Time

In [92]:
# total elapsed time
elapsed = time.time() - start
print(f"{elapsed:.2f}s to process {df_crime.shape[0]:,} rows")

83.32s to process 8,469,443 rows


## Save using PyArrow

In [93]:
# Save as Arrow
feather.write_feather(df_crime, "../Data/crime_data.feather")